In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
from scipy import io
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from sklearn import linear_model as lm

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

In [ ]:
power,coherence,granger,labels = load_data('/media/austin/ThickBoy__1/DataAgression/CL_baseline_all_validate2.mat',
                                          fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6 

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X = np.hstack((power,coherence,granger))


In [ ]:
windows = labels['windows']
windows.keys()

In [ ]:
mouse = np.squeeze(windows['mouse'])
expDate = np.squeeze(windows['expDate'])
group = np.squeeze(windows['group'])
condition = np.squeeze(windows['condition'])
behavior = np.squeeze(windows['behavior'])
time = np.squeeze(windows['time'])

In [ ]:
idx_pos = (condition==4)&(behavior==1)
indx_neg = (behavior==2)&((condition==4)|(condition==6)|(condition==8))
y = np.zeros(len(mouse))
y[idx_pos] = 1
idx_tot = idx_pos|indx_neg

In [ ]:
mice = np.unique(mouse)
nMice = len(mice)

## Pick subset of mice

In [ ]:
nTrain = 4
m_idx = rand.choice(len(mice),nTrain,replace=False)
tr_idx = np.zeros(nMice)
tr_idx[m_idx] = 1
mice_train = mice[tr_idx==1]
mice_test = mice[tr_idx==0]

In [ ]:
mice_train

### Get the training and test set

In [ ]:
training = np.zeros(len(mouse))
for i in range(nTrain):
    training[mouse==mice_train[i]] = 1

In [ ]:
np.sum(training)

## Now get training set

In [ ]:
X_train = X[training==1]
X_test = X[training==0]

In [ ]:
model_nmf = dp.NMF(30)
model_nmf.fit(X_train)

In [ ]:
Ex = np.mean(X_train,axis=0)
print(np.mean((X_train-Ex)**2))

In [ ]:
S_train = model_nmf.transform(X_train)
X_recon = np.dot(S_train,model_nmf.components_)
print(np.mean((X_train-X_recon)**2))
S_test = model_nmf.transform(X_test)
X_recon = np.dot(S_test,model_nmf.components_)
print(np.mean((X_test-X_recon)**2))

In [ ]:
#Ex = np.mean(X_test,axis=0)
print(np.mean((X_test-Ex)**2))

## Now see predictions

In [ ]:
idx_tot_train = idx_tot[training==1]
S_sub_train = S_train[idx_tot_train]
y_train = y[training==1]
y_sub_train = y_train[idx_tot_train]

In [ ]:
idx_tot_test = idx_tot[training==0]
S_sub_test = S_test[idx_tot_test]
y_test = y[training==0]
y_sub_test = y_test[idx_tot_test]

In [ ]:
model_lm = lm.LogisticRegressionCV()
model_lm.fit(S_sub_train,y_sub_train)

In [ ]:
y_pred = model_lm.decision_function(S_sub_test)

In [ ]:
roc_auc_score(y_sub_test,y_pred)

## Bayesian hierarchcial model

## Lets go through all the mice

In [ ]:
mouse_test = mouse[training==0]

In [ ]:
mouse_sub_test = mouse_test[idx_tot_test] 

In [ ]:
for i in range(4):
    yp = y_pred[mouse_sub_test==mice_test[i]]
    yt = y_sub_test[mouse_sub_test==mice_test[i]]
    print(mice_test[i],roc_auc_score(yt,yp))